In [36]:
import numpy as np
import pandas as pd
import folium
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)

point = [10.7767, 106.7031]
n = 500

lat = np.random.uniform(10.70, 10.86, n)
lon = np.random.uniform(106.60, 106.77, n)
hour = np.random.randint(0, 24, n)

demand = np.random.randint(20, 40, n)

central_area = (
    (lat > 10.755) & (lat < 10.815) &
    (lon > 106.675) & (lon < 106.735)
)

outer_area = ~central_area

early_morning = (hour >= 5) & (hour <= 6)
morning_peak = (hour >= 7) & (hour <= 9)
midday = (hour >= 10) & (hour <= 15)
evening_peak = (hour >= 17) & (hour <= 19)
night = (hour >= 20) & (hour <= 23)
late_night = (hour >= 0) & (hour <= 4)

demand = demand + 15 * early_morning
demand = demand + 40 * (outer_area & morning_peak)
demand = demand + 10 * midday
demand = demand + 45 * (central_area & evening_peak)
demand = demand + 20 * night
demand = demand - 5 * late_night

df = pd.DataFrame({
    "lat": lat,
    "lon": lon,
    "hour": hour,
    "demand": demand
})

X = df[["lat", "lon", "hour"]]
y = df["demand"]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

selected_hour = int(input("Nhập giờ muốn dự đoán (0-23): "))

df_predict = df[["lat", "lon"]].copy()
df_predict["hour"] = selected_hour
df_predict["predicted_demand"] = model.predict(df_predict[["lat", "lon", "hour"]])

def get_color(demand):
    if demand >= 55:
        return "red"
    elif demand >= 40:
        return "orange"
    else:
        return "green"

m = folium.Map(location=point, zoom_start=11)

for i in range(len(df_predict)):
    demand_value = df_predict.loc[i, "predicted_demand"]
    color = get_color(demand_value)

    folium.Circle(
        location=[df_predict.loc[i, "lat"], df_predict.loc[i, "lon"]],
        radius=120,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.5,
        popup="Nhu cầu dự đoán: " + str(round(demand_value, 2))
    ).add_to(m)

m


Nhập giờ muốn dự đoán (0-23): 14
